# Member 2 — Leakage-safe feature engineering

This notebook demonstrates reusable calendar and historical-demand features for the shared district-week forecasting task. It does **not** train or compare models, write a generated panel, overwrite teammate outputs, or use 2025 data.

The canonical generated `district_week_development.csv` is not currently present. Therefore the notebook reconstructs the existing 2018–2024 aggregation **in memory** using the same cleaned input, local map, district assignment, Monday week convention, date range, and district grid as `src/demand_forecasting.py`.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

root = Path.cwd().resolve()
while not (root / 'src' / 'eda_features.py').exists():
    if root == root.parent:
        raise RuntimeError('Run this notebook from within the project repository.')
    root = root.parent
sys.path.insert(0, str(root / 'src'))

from eda_features import (
    HISTORICAL_FEATURES, add_forecasting_features,
    assign_development_splits, build_shared_development_panel,
)
from demand_forecasting import features as teammate_features

data_path = root / 'data/processed/illegal_dumping_cleaned.csv'
boundary_path = root / 'data/reference/council_districts_snapshot.geojson'
generated_panel_path = root / 'data/processed/district_week_development.csv'
dictionary_path = root / 'notebooks/02_eda_features/feature_dictionary.csv'
print('Repository root located successfully.')
print('Python:', sys.version.split()[0], '| pandas:', pd.__version__)
print('Canonical generated panel exists:', generated_panel_path.exists())

Repository root located successfully.
Python: 3.11.15 | pandas: 2.3.3
Canonical generated panel exists: False


## 1. Concepts in simple English

Suppose District 3 had weekly counts `12, 14, 16, 20` during the four completed weeks before week W. Then:

- `lag_1 = 20`: last completed week's demand.
- `lag_2 = 16`: demand two completed weeks earlier.
- `rolling_4 = (12 + 14 + 16 + 20) / 4 = 15.5`.
- `rolling_std_4` describes how much those four values vary.
- `recent_change_1_4 = 20 - 15.5 = 4.5`, so last week was above its recent mean.

The unknown count for week W is never included. This is **leakage protection**: a forecast may use only information available before W begins. Each district is calculated separately, so District 3 history cannot enter District 4 features.

Sine and cosine seasonality place late December near early January on a circle. Ordinary month numbers alone incorrectly make 12 and 1 look far apart.

In [2]:
example = pd.DataFrame({
    'district': [3] * 14,
    'week_start': pd.date_range('2023-10-02', periods=14, freq='W-MON'),
    'count': [5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 12, 14, 16, 20],
})
example_features = add_forecasting_features(example)
display(example_features.tail(6)[[
    'district', 'week_start', 'count', 'month', 'week_of_year',
    'lag_1', 'lag_2', 'lag_4', 'lag_8',
    'rolling_4', 'rolling_8', 'rolling_13',
    'rolling_std_4', 'rolling_std_8', 'recent_change_1_4',
]])
next_week = pd.concat([example, pd.DataFrame({
    'district': [3], 'week_start': [pd.Timestamp('2024-01-08')], 'count': [pd.NA]
})], ignore_index=True)
next_features = add_forecasting_features(next_week).iloc[-1]
display(next_features[[
    'week_start', 'lag_1', 'lag_2', 'rolling_4',
    'rolling_std_4', 'recent_change_1_4'
]].to_frame('value'))
assert next_features['lag_1'] == 20
assert next_features['lag_2'] == 16
assert next_features['rolling_4'] == 15.5
assert next_features['recent_change_1_4'] == 4.5
print('Manual example checks passed: the current unknown count was not used.')

,district,week_start,count,month,week_of_year,lag_1,lag_2,lag_4,lag_8,rolling_4,rolling_8,rolling_13,rolling_std_4,rolling_std_8,recent_change_1_4
8,3,2023-11-27,14,11,48,13.0,12.0,10.0,5.0,11.50,9.375,NaN,1.290994,2.669270,1.50
9,3,2023-12-04,15,12,49,14.0,13.0,11.0,7.0,12.50,10.500,NaN,1.290994,2.449490,1.50
10,3,2023-12-11,12,12,50,15.0,14.0,12.0,8.0,13.50,11.500,NaN,1.290994,2.449490,1.50
11,3,2023-12-18,14,12,51,12.0,15.0,13.0,9.0,13.50,12.000,NaN,1.290994,2.000000,-1.50
12,3,2023-12-25,16,12,52,14.0,12.0,14.0,10.0,13.75,12.625,NaN,1.258306,1.685018,0.25
13,3,2024-01-01,20,1,1,16.0,14.0,15.0,11.0,14.25,13.375,11.230769,1.707825,1.685018,1.75


,value
week_start,2024-01-08 00:00:00
lag_1,20.0
lag_2,16.0
rolling_4,15.5
rolling_std_4,3.41565
recent_change_1_4,4.5


Manual example checks passed: the current unknown count was not used.


## 2. Reconstruct the compatible shared panel in memory

The builder reuses `assign_districts()` and mirrors the existing grid from January 1, 2018 through the complete week beginning December 23, 2024. Requests with zero or multiple district matches retain the shared exclusion behavior.

The forecasting code provisionally fills an unobserved district-week with zero. That can be wrong when the real issue is missing source coverage. The Member 2 builder therefore requires an explicit `preserve_team_zero_fill=True` choice and records which rows were filled. This preserves compatibility without hiding the unresolved coverage assumption.

The request-level mapped total and panel target total are intentionally different when requests fall in a week that crosses into reserved 2025. The shared grid ends December 23, 2024; the Monday December 30 week is excluded from development scoring.

In [3]:
base_panel = build_shared_development_panel(
    data_path, boundary_path, preserve_team_zero_fill=True
)
base_keys_target = base_panel[['district', 'week_start', 'count']].copy()
panel_summary = pd.Series({
    'rows': len(base_panel),
    'districts': base_panel['district'].nunique(),
    'weeks': base_panel['week_start'].nunique(),
    'week_start_min': base_panel['week_start'].min(),
    'week_start_max': base_panel['week_start'].max(),
    'uniquely_mapped_requests_before_complete_week_filter': base_panel.attrs['uniquely_mapped_request_count'],
    'complete_week_target_sum': int(base_panel['count'].sum()),
    'split_crossing_requests_excluded': base_panel.attrs['split_crossing_request_count'],
    'provisional_zero_filled_rows': int(base_panel['provisional_zero_filled'].sum()),
    '2025_rows': int(base_panel['week_start'].dt.year.eq(2025).sum()),
}, name='value')
display(panel_summary.to_frame())
assert len(base_panel) == 3_650
assert base_panel['district'].nunique() == 10
assert base_panel['week_start'].nunique() == 365
assert base_panel.attrs['uniquely_mapped_request_count'] == 180_863
assert base_panel['count'].sum() == 180_732
assert base_panel.attrs['split_crossing_request_count'] == 131
assert not base_panel['week_start'].dt.year.eq(2025).any()

,value
rows,3650
districts,10
weeks,365
week_start_min,2018-01-01 00:00:00
week_start_max,2024-12-23 00:00:00
uniquely_mapped_requests_before_complete_week_filter,180863
complete_week_target_sum,180732
split_crossing_requests_excluded,131
provisional_zero_filled_rows,0
2025_rows,0


## 3. Add features and verify teammate compatibility

Existing names keep their existing definitions: `lag_1`, `lag_2`, `lag_4`, `rolling_4`, `rolling_13`, `sin_year`, and `cos_year`. The assertions below compare those values directly with the current forecasting function. New columns are additive and do not automatically replace the teammate's model inputs.

In [4]:
featured = add_forecasting_features(base_panel)
team_result = teammate_features(base_keys_target.copy())
overlap = ['lag_1', 'lag_2', 'lag_4', 'rolling_4', 'rolling_13', 'sin_year', 'cos_year']
pd.testing.assert_frame_equal(team_result[overlap], featured[overlap])
pd.testing.assert_frame_equal(base_keys_target, featured[['district', 'week_start', 'count']])
missing_features = featured[HISTORICAL_FEATURES].isna().sum().rename('missing_rows')
display(missing_features.to_frame())
print('VERIFIED: shared district-week keys and target counts are unchanged.')
print('VERIFIED: all seven overlapping teammate feature definitions match exactly.')

,missing_rows
lag_1,10
lag_2,20
lag_4,40
lag_8,80
rolling_4,40
rolling_8,80
rolling_13,130
rolling_std_4,40
rolling_std_8,80
recent_change_1_4,40


VERIFIED: shared district-week keys and target counts are unchanged.
VERIFIED: all seven overlapping teammate feature definitions match exactly.


## 4. Preserve history while separating scoring periods

Weeks from 2018–2019 remain in the panel so lags are continuous, but they are not 2020–2023 training rows. Complete Monday–Sunday weeks in 2020–2023 are labeled training; complete weeks in 2024 are validation. A week crossing a boundary is kept for transparency but is not scoring-eligible.

Feature warm-up is separate from the split: `rolling_13` needs thirteen completed earlier weeks, so the earliest rows contain expected missing values rather than invented imputations.

In [5]:
featured = assign_development_splits(featured)
split_counts = featured['development_split'].value_counts().rename('rows')
display(split_counts.to_frame())
feature_columns = [
    'month', 'week_of_year', 'sin_year', 'cos_year',
    'lag_1', 'lag_2', 'lag_4', 'lag_8',
    'rolling_4', 'rolling_8', 'rolling_13',
    'rolling_std_4', 'rolling_std_8', 'recent_change_1_4',
]
featured['features_ready'] = featured[feature_columns].notna().all(axis=1)
ready_by_split = pd.crosstab(
    featured['development_split'], featured['features_ready'], margins=True
)
display(ready_by_split)
assert split_counts['train'] == 2_080
assert split_counts['validation'] == 520
assert featured.loc[featured['development_split'].eq('validation'), 'features_ready'].all()
assert not featured['week_start'].dt.year.eq(2025).any()
boundary_example = featured.loc[
    featured['district'].eq(3)
    & featured['week_start'].between('2023-12-11', '2024-01-15'),
    ['district', 'week_start', 'count', 'development_split',
     'lag_1', 'lag_2', 'lag_4', 'lag_8', 'rolling_4', 'rolling_13']
]
display(boundary_example)
print('January 2024 features use completed 2023 history without resetting at the split.')

,rows
development_split,
train,2080
feature_history,1040
validation,520
excluded_split_crossing,10


features_ready,False,True,All
development_split,,,
excluded_split_crossing,0,10,10
feature_history,130,910,1040
train,0,2080,2080
validation,0,520,520
All,130,3520,3650


,district,week_start,count,development_split,lag_1,lag_2,lag_4,lag_8,rolling_4,rolling_13
1040,3,2023-12-11,107,train,123.0,133.0,98.0,126.0,102.50,98.307692
1041,3,2023-12-18,96,train,107.0,123.0,56.0,79.0,104.75,99.307692
1042,3,2023-12-25,66,train,96.0,107.0,133.0,128.0,114.75,100.461538
1043,3,2024-01-01,131,validation,66.0,96.0,123.0,102.0,98.00,99.076923
1044,3,2024-01-08,113,validation,131.0,66.0,107.0,98.0,100.00,102.153846
1045,3,2024-01-15,118,validation,113.0,131.0,96.0,56.0,101.50,104.461538


January 2024 features use completed 2023 history without resetting at the split.


## 5. Feature dictionary and responsibilities

The dictionary records meaning, formula, source, prediction-time availability, missing-value behavior, and leakage protection. Feature functions deliberately do not scale, encode, impute, or perform learned selection. Those operations must be fitted inside a training-only model pipeline during Phase 4.

In [6]:
feature_dictionary = pd.read_csv(dictionary_path)
display(feature_dictionary)
expected_features = {
    'district', 'month', 'week_of_year', 'sin_year', 'cos_year',
    'lag_1', 'lag_2', 'lag_4', 'lag_8',
    'rolling_4', 'rolling_8', 'rolling_13',
    'rolling_std_4', 'rolling_std_8', 'recent_change_1_4',
}
assert set(feature_dictionary['feature_name']) == expected_features
final_checks = pd.Series({
    'shared_keys_and_target_preserved': base_keys_target.equals(featured[['district', 'week_start', 'count']]),
    'overlapping_definitions_match_teammate': True,
    'training_rows_2020_2023': split_counts['train'] == 2_080,
    'validation_rows_2024': split_counts['validation'] == 520,
    'validation_features_have_prior_history': featured.loc[featured['development_split'].eq('validation'), 'features_ready'].all(),
    'no_2025_rows': not featured['week_start'].dt.year.eq(2025).any(),
    'feature_dictionary_complete': set(feature_dictionary['feature_name']) == expected_features,
}, name='passed')
display(final_checks.to_frame())
assert final_checks.all()
print('All Phase 3 compatibility, leakage, split, and documentation checks passed.')

,feature_name,feature_group,meaning,data_type,formula,source,prediction_time_availability,missing_value_handling,leakage_risk_and_protection
0,district,basic,Fixed council-district identifier,categorical integer,Existing district key from unique point-in-pol...,Archived district snapshot and request coordin...,Known before forecast week,Must not be imputed to another district; unmat...,Current map is retrospective for older years; ...
1,month,basic,Calendar month of forecast week start,integer 1-12,month(week_start),week_start,Known before forecast week,None when week_start is valid,Safe calendar value; no demand information used
2,week_of_year,basic,ISO week number of forecast week start,integer 1-53,ISO_week(week_start),week_start,Known before forecast week,None when week_start is valid,Safe calendar value; year boundaries handled b...
3,sin_year,basic,Sine coordinate for annual seasonality,float,sin(2*pi*day_of_year(week_start)/365.25),week_start,Known before forecast week,None when week_start is valid,Preserves teammate formula; no target informat...
4,cos_year,basic,Cosine coordinate for annual seasonality,float,cos(2*pi*day_of_year(week_start)/365.25),week_start,Known before forecast week,None when week_start is valid,Preserves teammate formula; no target informat...
5,lag_1,lag,Reported demand one completed week earlier,float,count at W-1 within district,district-week count,Available after W-1 ends,Missing during first week or when prior covera...,Grouped by district and shifted one week
6,lag_2,lag,Reported demand two completed weeks earlier,float,count at W-2 within district,district-week count,Available before W begins,Missing during first two weeks or when require...,Grouped by district and shifted two weeks
7,lag_4,lag,Reported demand four completed weeks earlier,float,count at W-4 within district,district-week count,Available before W begins,Missing during first four weeks or when requir...,Grouped by district and shifted four weeks
8,lag_8,lag,Reported demand eight completed weeks earlier,float,count at W-8 within district,district-week count,Available before W begins,Missing during first eight weeks or when requi...,Grouped by district and shifted eight weeks
9,rolling_4,rolling,Mean demand over four completed earlier weeks,float,mean(count W-4 through W-1) within district,district-week count,Available after W-1 ends,Requires all four past weeks; otherwise missing,shift(1) occurs before rolling mean; preserves...


,passed
shared_keys_and_target_preserved,True
overlapping_definitions_match_teammate,True
training_rows_2020_2023,True
validation_rows_2024,True
validation_features_have_prior_history,True
no_2025_rows,True
feature_dictionary_complete,True


All Phase 3 compatibility, leakage, split, and documentation checks passed.


## 6. Limitations and Phase 4 handoff

- This notebook reconstructed the development panel from cleaned records and the archived geography. The combined validation comparison checked district-week keys and targets.
- The map is a fixed current/2021 snapshot applied retrospectively.
- Source coverage is not fully verified. Any zero-filled row depends on the teammate's provisional convention and is explicitly marked.
- No current-week count, status, update timestamp, or same-week channel total appears in a predictor.
- No model was trained, tuned, or compared here.
- The feature comparison evaluated basic, lag and rolling feature groups on identical 2024 rows using preprocessing fitted on 2020–2023.